# RL Inspection Planner

This notebook builds the reinforcement learning simulation part of the autonomous concrete infrastructure inspection robot prototype. The goal is to learn an inspection route over a simulated concrete surface.

The RL planner is a grid-world simulation and does not control a real robot in this version.

In [ ]:
# pathlib creates clean project paths.
from pathlib import Path

# sys lets this notebook import modules from src/.
import sys

# defaultdict creates empty Q-table rows automatically.
from collections import defaultdict

# numpy handles arrays and random risk-map generation.
import numpy as np

# pandas displays risk maps and route logs as tables.
import pandas as pd

# matplotlib creates and saves plots.
import matplotlib.pyplot as plt

# PIL displays saved output images inside the notebook.
from PIL import Image

# display renders pandas tables cleanly.
from IPython.display import display

# Resolve the project root from either the root or notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add the project root so src/ imports work.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import reusable visualization helpers.
from src.visualization import plot_crack_risk_map, plot_inspection_route

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## Grid-World Inspection Idea

The simulated concrete surface is a square grid. Each cell has a crack-risk score between 0 and 1. The robot can move up, down, left, right, or inspect the current cell.

In [ ]:
# Grid and Q-learning configuration.
GRID_SIZE = 6
NUM_EPISODES = 1000
LEARNING_RATE = 0.1
DISCOUNT_FACTOR = 0.95
EPSILON = 0.30
RANDOM_SEED = 42

print("RL configuration")
print(f"GRID_SIZE: {GRID_SIZE} x {GRID_SIZE}")
print(f"NUM_EPISODES: {NUM_EPISODES}")
print(f"LEARNING_RATE: {LEARNING_RATE}")
print(f"DISCOUNT_FACTOR: {DISCOUNT_FACTOR}")
print(f"EPSILON: {EPSILON}")
print(f"RANDOM_SEED: {RANDOM_SEED}")

## Simulated Concrete Inspection Surface

In a real system, the vision model would estimate crack risk from camera images. Here we simulate that input with a random risk map containing a few high-risk hotspots.

In [ ]:
# Create a reproducible random generator.
rng = np.random.default_rng(RANDOM_SEED)

# Most cells start with low-to-medium crack risk.
risk_map = rng.uniform(0.05, 0.45, size=(GRID_SIZE, GRID_SIZE))

# Add high-risk hotspots that represent likely crack regions.
hotspot_count = max(3, GRID_SIZE // 2)
hotspot_indices = rng.choice(GRID_SIZE * GRID_SIZE, size=hotspot_count, replace=False)
for index in hotspot_indices:
    row, col = divmod(int(index), GRID_SIZE)
    risk_map[row, col] = rng.uniform(0.75, 1.0)

# Print the risk map as a readable table.
risk_map_df = pd.DataFrame(
    np.round(risk_map, 2),
    index=[f"row_{i}" for i in range(GRID_SIZE)],
    columns=[f"col_{j}" for j in range(GRID_SIZE)],
)

print("Simulated crack risk map values:")
display(risk_map_df)

In [ ]:
# Visualize and save the crack risk map.
risk_map_path = plot_crack_risk_map(
    risk_map=risk_map,
    output_path=OUTPUT_DIR / "crack_risk_map.png",
    title="Simulated Crack Risk Map",
)

# Display the saved heatmap.
saved_risk_map = Image.open(risk_map_path)
plt.figure(figsize=(7, 6))
plt.imshow(saved_risk_map)
plt.axis("off")
plt.show()

print(f"Saved crack risk map to: {risk_map_path}")

## State, Action, and Reward Design

State includes robot position, whether the current cell is high-risk, and inspected-cell history.

Actions: up, down, left, right, inspect.

Rewards: `+10` for inspecting a high-risk crack cell, `-1` movement cost, `-2` repeated inspection, `-5` invalid movement, and `+20` for completing inspection of all high-risk cells.

In [ ]:
class InspectionGridEnvironment:
    """Grid-world environment for simulated concrete inspection."""

    ACTION_NAMES = ["up", "down", "left", "right", "inspect"]

    def __init__(self, risk_map, high_risk_threshold=0.65, start_position=(0, 0)):
        # Store the risk map and basic grid settings.
        self.risk_map = np.asarray(risk_map, dtype=float)
        self.grid_size = self.risk_map.shape[0]
        self.high_risk_threshold = high_risk_threshold
        self.start_position = start_position
        self.max_steps = self.grid_size * self.grid_size * 3
        self.reset()

    def reset(self):
        # Start each episode at the start position with no inspected cells.
        self.position = self.start_position
        self.inspected = np.zeros_like(self.risk_map, dtype=bool)
        self.steps = 0
        self.done = False
        return self._get_state()

    def _get_state(self):
        # State includes position, local high-risk flag, and inspected-cell mask.
        row, col = self.position
        current_risk_is_high = int(self.risk_map[row, col] >= self.high_risk_threshold)
        inspected_mask = tuple(self.inspected.astype(int).flatten().tolist())
        return (row, col, current_risk_is_high, inspected_mask)

    def _all_high_risk_cells_inspected(self):
        # Check whether every high-risk cell has been inspected.
        high_risk_cells = self.risk_map >= self.high_risk_threshold
        return bool(np.all(self.inspected[high_risk_cells]))

    def step(self, action):
        # If the episode is finished, return the current state immediately.
        if self.done:
            return self._get_state(), 0.0, True, {"message": "episode already done"}

        row, col = self.position
        reward = 0.0
        info = {"action_name": self.ACTION_NAMES[action]}

        # Movement actions propose a next cell and receive movement cost.
        if action == 0:
            next_position, reward = (row - 1, col), -1.0
        elif action == 1:
            next_position, reward = (row + 1, col), -1.0
        elif action == 2:
            next_position, reward = (row, col - 1), -1.0
        elif action == 3:
            next_position, reward = (row, col + 1), -1.0
        elif action == 4:
            next_position = (row, col)
            if self.inspected[row, col]:
                reward = -2.0
                info["inspection"] = "repeated"
            else:
                self.inspected[row, col] = True
                risk = self.risk_map[row, col]
                reward = 10.0 if risk >= self.high_risk_threshold else 0.0
                info["inspection"] = "new"
                info["risk"] = float(risk)
        else:
            raise ValueError(f"Unknown action: {action}")

        # Apply movement only if the next cell is inside the grid.
        if action in [0, 1, 2, 3]:
            next_row, next_col = next_position
            if 0 <= next_row < self.grid_size and 0 <= next_col < self.grid_size:
                self.position = next_position
            else:
                reward = -5.0
                info["movement"] = "invalid"

        # Add completion bonus when all high-risk cells are inspected.
        if self._all_high_risk_cells_inspected():
            reward += 20.0
            self.done = True
            info["completion_bonus"] = True

        # Stop very long episodes.
        self.steps += 1
        if self.steps >= self.max_steps:
            self.done = True
            info["time_limit"] = True

        return self._get_state(), reward, self.done, info

In [ ]:
# Test the environment using a few random actions.
test_env = InspectionGridEnvironment(risk_map=risk_map, high_risk_threshold=0.65)
state = test_env.reset()
print(f"Initial state: row={state[0]}, col={state[1]}, high_risk_flag={state[2]}")

for step_index in range(8):
    # Pick a random action only for this sanity check.
    action = int(rng.integers(len(test_env.ACTION_NAMES)))
    next_state, reward, done, info = test_env.step(action)

    print(
        f"Step {step_index + 1}: "
        f"action={info['action_name']}, "
        f"position={test_env.position}, "
        f"reward={reward}, done={done}, info={info}"
    )

    if done:
        break

## Q-Learning

Q-learning stores action values in a Q-table. The agent explores random actions early in training, then gradually relies more on actions with high learned value.

In [ ]:
class QLearningAgent:
    """Tabular Q-learning agent with epsilon-greedy action selection."""

    def __init__(self, num_actions, learning_rate, discount_factor, epsilon, random_seed=42):
        # Store hyperparameters and initialize exploration settings.
        self.num_actions = num_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.min_epsilon = 0.02
        self.epsilon_decay = 0.995
        self.rng = np.random.default_rng(random_seed)

        # New states receive one Q-value per action, initialized to zero.
        self.q_table = defaultdict(lambda: np.zeros(self.num_actions, dtype=float))

    def select_action(self, state, training=True):
        # Explore randomly during training with probability epsilon.
        if training and self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.num_actions))

        # Choose a best-valued action, with random tie-breaking.
        q_values = self.q_table[state]
        best_actions = np.flatnonzero(np.isclose(q_values, q_values.max()))
        return int(self.rng.choice(best_actions))

    def update(self, state, action, reward, next_state, done):
        # Current Q-value estimate.
        old_q = self.q_table[state][action]

        # If done, future value is zero; otherwise use the best next action value.
        best_next_q = 0.0 if done else np.max(self.q_table[next_state])

        # Q-learning update rule.
        target_q = reward + self.discount_factor * best_next_q
        self.q_table[state][action] = old_q + self.learning_rate * (target_q - old_q)

    def decay_exploration(self):
        # Reduce random exploration after each episode.
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

In [ ]:
# Train the Q-learning agent.
env = InspectionGridEnvironment(risk_map=risk_map, high_risk_threshold=0.65)
agent = QLearningAgent(
    num_actions=len(env.ACTION_NAMES),
    learning_rate=LEARNING_RATE,
    discount_factor=DISCOUNT_FACTOR,
    epsilon=EPSILON,
    random_seed=RANDOM_SEED,
)

episode_rewards = []
progress_interval = max(1, NUM_EPISODES // 10)

for episode in range(1, NUM_EPISODES + 1):
    # Reset the environment at the start of each episode.
    state = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        # Select an action, step the environment, and update the Q-table.
        action = agent.select_action(state, training=True)
        next_state, reward, done, info = env.step(action)
        agent.update(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    # Decay exploration and store total episode reward.
    agent.decay_exploration()
    episode_rewards.append(total_reward)

    # Print training progress periodically.
    if episode % progress_interval == 0:
        print(
            f"Episode {episode:4d}/{NUM_EPISODES} | "
            f"reward={total_reward:7.2f} | epsilon={agent.epsilon:.3f} | "
            f"q_states={len(agent.q_table)}"
        )

In [ ]:
# Plot episode rewards and a rolling average.
reward_series = pd.Series(episode_rewards)
rolling_rewards = reward_series.rolling(window=50, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(reward_series.values, alpha=0.35, label="Episode reward")
ax.plot(rolling_rewards.values, linewidth=2, label="50-episode rolling average")
ax.set_title("Q-learning Training Rewards")
ax.set_xlabel("Episode")
ax.set_ylabel("Total reward")
ax.legend()
fig.tight_layout()

rl_rewards_path = OUTPUT_DIR / "rl_training_rewards.png"
fig.savefig(rl_rewards_path, dpi=150)
plt.show()

print(f"Saved RL training rewards plot to: {rl_rewards_path}")

## Learned Route Generation

After training, run the agent without exploration to generate an inspection route. The route should favor reaching and inspecting high-risk cells.

In [ ]:
# Generate a route using the learned Q-table.
route_env = InspectionGridEnvironment(risk_map=risk_map, high_risk_threshold=0.65)
state = route_env.reset()
route = [route_env.position]
route_log = []

done = False
step_index = 0
while not done:
    # Select the greedy learned action.
    action = agent.select_action(state, training=False)
    next_state, reward, done, info = route_env.step(action)

    # Store route information for display.
    route.append(route_env.position)
    route_log.append(
        {
            "step": step_index,
            "action": info["action_name"],
            "position_after_action": route_env.position,
            "reward": reward,
            "done": done,
        }
    )

    state = next_state
    step_index += 1

route_df = pd.DataFrame(route_log)
print("Learned inspection route steps:")
display(route_df)

high_risk_mask = risk_map >= 0.65
inspected_high_risk = int(route_env.inspected[high_risk_mask].sum())
total_high_risk = int(high_risk_mask.sum())
print(f"High-risk cells inspected: {inspected_high_risk}/{total_high_risk}")

In [ ]:
# Visualize and save the learned inspection route.
inspection_route_path = plot_inspection_route(
    risk_map=risk_map,
    route=route,
    output_path=OUTPUT_DIR / "inspection_route.png",
    inspected_cells=route_env.inspected,
    title="Learned Inspection Route",
)

# Display the saved route image.
saved_route_image = Image.open(inspection_route_path)
plt.figure(figsize=(7, 7))
plt.imshow(saved_route_image)
plt.axis("off")
plt.show()

print(f"Saved inspection route visualization to: {inspection_route_path}")

## Connection to the Crack Detection Model

The vision model estimates crack probability from concrete images. Those probabilities can populate a crack-risk map. The RL planner then uses the risk map to decide where to inspect next.

In [ ]:
# Print a final summary of generated RL output files.
rl_output_files = [
    OUTPUT_DIR / "crack_risk_map.png",
    OUTPUT_DIR / "rl_training_rewards.png",
    OUTPUT_DIR / "inspection_route.png",
]

print("Generated RL output files:")
for path in rl_output_files:
    print(f"  {path} | exists={path.exists()}")